In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.mask import decode
import os

# 1. 读取原始图像
# image_path = 'path/to/your/image.jpg'  # 原始图像路径
# image = cv2.imread(image_path)
# image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # 转换为 RGB 格式

# 2. 解析 JSON 文件
image_folder = 'val2017'
json_path = 'sem_seg_predictions_1.json'  # JSON 文件路径
with open(json_path) as f:
    data = json.load(f)

output_folder = 'overlay_images'  # 保存结果的文件夹路径

# 创建输出文件夹（如果不存在）
os.makedirs(output_folder, exist_ok=True)

# 读取 JSON 文件
with open(json_path) as f:
    data = json.load(f)

color_map = {
    1: np.array([255, 0, 0]),  # 红色
    2: np.array([0, 255, 0]),  # 绿色
}


# 遍历文件夹中的所有图像
for filename in os.listdir(image_folder):
    if filename.endswith(('.jpg', '.jpeg', '.png')):  # 只处理图像文件
        image_path = os.path.join(image_folder, filename)
        
        # 读取原始图像
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # 转换为 RGB 格式

        # 处理对应的分割结果
        # 如果是背景，就不显示
        for result in data:
            if result['file_name'].endswith(filename):  # 匹配当前图像的分割结果
                segmentation = result['segmentation']
                category_id = result['category_id']

                if category_id != 0:
                    # 使用 RLE 解码S
                    rle = {
                        'size': segmentation['size'],
                        'counts': segmentation['counts']
                    }
                    mask = decode(rle)  # 解码得到二进制掩码（0和1）

                    # 同一个种类始终使用同一个颜色
                    color = color_map.get(category_id, (255, 255, 255))

                    # 生成掩码区域的颜色
                    image[mask == 1] = image[mask == 1] * 0.5 + color * 0.5  # 半透明叠加

        # 保存结果图像
        output_path = os.path.join(output_folder, filename)
        cv2.imwrite(output_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))